In [1]:
%reload_kedro

[06/26/25 09:43:26] INFO     Resolved project path as:                                              ]8;id=919968;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=10496;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\ipython\__init__.py#175\175]8;;\
                             C:\Users\nicolas.betancourt\Documents\GitHub\pytorch\serpientes-de-col                
                             ombia.                                                                                
                             To set a different path, run '%reload_kedro <project_root>'                           

                    INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=667867;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro_telemetry\plugin.py\plugin.py]8;;\:]8;id=710425;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro_telemetry\plugin.py#233\233]8;;\
                             the product. No personal data or IP addresses are stored on our side. If              
                             you want to opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK`              
                             environment variables, or create a `.telemetry` file in the current                   
                             working directory with the contents `consent: false`. Read more at                    
                             https://docs.kedro.org/en/stable/configuration/telemetry.html                         

[06/26/25 09:43:27] INFO     Kedro project serpientes_de_colombia                                   ]8;id=12782;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=859993;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\ipython\__init__.py#141\141]8;;\

                    INFO     Defined global variable 'context', 'session', 'catalog' and            ]8;id=869594;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=966895;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\ipython\__init__.py#142\142]8;;\
                             'pipelines'                                                                           

                    INFO     Registered line magic 'run_viz'                                        ]8;id=925012;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=588081;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\ipython\__init__.py#148\148]8;;\

In [2]:
import torch
from torch.utils.data import Dataset,  DataLoader
from PIL import Image
import os
from torchvision import  transforms
import matplotlib.pyplot as plt
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

In [3]:
import torch
print(torch.cuda.is_available())

True


In [4]:
train_data=catalog.load('train_image_dataset')

                    INFO     Loading data from train_image_dataset                              ]8;id=690825;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=47216;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\io\data_catalog.py#539\539]8;;\
                             (KedroPytorchImageDataset)...                                                         

In [5]:
training_params = catalog.load('params:dense_params')
label_map=catalog.load('label_encoder')

                    INFO     Loading data from params:dense_params (MemoryDataset)...           ]8;id=300799;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=443676;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\io\data_catalog.py#539\539]8;;\

                    INFO     Loading data from label_encoder (JSONDataset)...                   ]8;id=847752;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=491636;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\Lib\site-packages\kedro\io\data_catalog.py#539\539]8;;\

# Custom dataset class

In [7]:
transform=transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
target_transform=lambda x: label_map.get(x)
training_set = train_data.with_transforms(transform=transform, target_transform=target_transform) #CustomDataset(data=train_data.data, transform=transform)

In [8]:
training_set.transform

Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

In [9]:
training_generator = DataLoader(training_set, **training_params)


# Model

In [11]:
import os
from torch import nn

In [12]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [13]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(3*224*224, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 2),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [14]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=150528, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=2, bias=True)
  )
)


# Training

In [16]:
learning_rate = 1e-3
batch_size = 64
epochs = 5
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [36]:
size = len(training_generator.dataset)
# Set the model to training mode - important for batch normalization and dropout layers
# Unnecessary in this situation but added for best practices
model.train()
total_loss=0

train_loss_list=[]
test_loss_list=[]
test_accuracy_list=[]

for batch, (X, y) in enumerate(training_generator):
    X, y = X.to(device), y.to(device)
    # Compute prediction and loss
    pred = model(X)
    loss = loss_fn(pred, y)

    # Backpropagation
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    total_loss += loss.item()

    if batch % 10 == 0:
        loss, current = loss.item(), batch * batch_size + len(X)
        print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
avg_loss = total_loss / len(training_generator)

loss: 0.711594  [   15/  317]
loss: 0.563318  [  655/  317]
loss: 0.633756  [ 1295/  317]
